# Alert Dispatcher

Forwards pipeline failure CloudEvents from Fabric Activator to Azure Storage Queue.
Triggered by the activator's action pipeline — receives the event payload as a parameter.

In [ ]:
# Parameters — injected by the pipeline at runtime
event_payload = '{}'  # CloudEvent JSON string from activator
queue_scope = 'source'  # 'source' or 'domain' — routes to the correct queue

In [ ]:
# Install azure-storage-queue SDK (available via workspace managed identity)
!pip install azure-storage-queue azure-identity -q

In [ ]:
import json
import base64
import time
from azure.storage.queue import QueueClient
from azure.core.credentials import AccessToken

STORAGE_ACCOUNT = 'stvdstudiosharedfiles'
QUEUE_NAMES = {
    'source': 'pipeline-alerts-source',
    'domain': 'pipeline-alerts-domain',
}

if not event_payload or event_payload == '{}':
    raise ValueError('event_payload parameter is empty')

queue_name = QUEUE_NAMES.get(queue_scope)
if not queue_name:
    raise ValueError(f'Invalid queue_scope: {queue_scope}')

event = json.loads(event_payload)
print(f'Dispatching event {event.get("id", "unknown")} to queue: {queue_name}')

class FabricTokenCredential:
    def get_token(self, *args, **kwargs):
        token = notebookutils.credentials.getToken(audience="storage")
        return AccessToken(token, int(time.time()) + 600)

credential = FabricTokenCredential()
queue_client = QueueClient(
    account_url=f'https://{STORAGE_ACCOUNT}.queue.core.windows.net',
    queue_name=queue_name,
    credential=credential,
)

message_bytes = json.dumps(event).encode('utf-8')
message_b64 = base64.b64encode(message_bytes).decode('utf-8')

queue_client.send_message(message_b64)
print(f'Event dispatched successfully to {queue_name}')